# Run the STURM ⇄ MESSAGEix-Austria linkage — results batch

Open this notebook from the repository root. It needs the bundled assets in place (`src/sturm/` and the two workbooks in `data/` — see `data/README.md`), a full **GAMS** licence, and **R** (`Rscript`).

This notebook produces **only the thesis results**: the 3×3 pathway × digitalization matrix, the linear fossil-exit sensitivity, and the carbon-price probe, then tables and figures. Expected wall time for a fresh batch: **~1.2–1.8 h**. Everything diagnostic (STURM smoke test, bounds-mode sensitivity comparison, once-through, verification checks) lives in `diagnostics.ipynb` — run on demand.

**Coupling: the Mastrucci soft coupling in useful energy (the only mode since the 2026-09-13 prune)** (STURM final energy → useful via the techs' base-year input coefficients → `rc_therm` demand; fuel mix optimised subject to the base-year calibration pin + one-sided pathway caps). The r5 bounds-mode canon is the hard-linkage sensitivity — **before this batch archive it**: `mv results/runs results/runs_r5_bounds`.

Runtime options come from the `config` sheet of `data/inputs.xlsx` (edit there — the workbook is the single source of every tunable parameter). Outputs go to `results/runs/<scenario-id>/`; the full run log to `results/logs/`.

**Convergence protocol (W5.8/W5.8b):** the price feedback is under-relaxed (`price_damping = 0.5`, budget 20). Run the §2 probes first and check both print `Linkage converged …`. The batch cells register every run; §3i then automatically re-runs *only* the cells whose runlog verdict is non-converged, at `price_damping = 0.3` with a doubled budget (40) — per-cell α is recorded in `convergence_evidence.csv`.

In [ ]:
import logging, os, sys
from pathlib import Path

# --- ixmp platform. Set $SMX_PLATFORM to use a private database (recommended on a
# shared server: the shared default store corrupts under repeated open/close):
#   ixmp platform add <name> jdbc hsqldb /home/<you>/<name>-localdb/default
#   export SMX_PLATFORM=<name>
# Unset → the workbook's platform_name (blank = the ixmp local default database).
# NOTE: load_base_scenario imports the baseline FRESH for every cell (~25 s each;
# no state leaks between cells).
import ixmp
_plat = os.environ.get("SMX_PLATFORM")
if _plat:
    try:
        ixmp.config.get_platform_info(_plat)
    except ValueError:  # register it (ixmp config is per-environment, not part of the repo)
        _db = Path.home() / f"{_plat}-localdb" / "default"
        ixmp.config.add_platform(_plat, "jdbc", "hsqldb", str(_db))
        ixmp.config.save()
        print(f"registered ixmp platform {_plat!r} -> {_db}")

# make the package importable without `pip install -e .`
src = Path.cwd() / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
logging.basicConfig(level=logging.INFO, force=True)

from common.config import Config
from linkage.pipeline import run_scenario, run_scenarios
from messageix import build
from common.scenarios import Scenario, pathways, digitalization_levels
from linkage.runlog import start as start_runlog

# tee every INFO log (convergence trace, STURM year summary, warnings) to a file
LOGFILE = start_runlog()
print("run log ->", LOGFILE)

cfg = Config.load()   # reads data/inputs.xlsx (the single input source; no fallbacks)
# r6 re-solve (2026-07-14, changelog §17): the headline coupling flips to the
# Mastrucci soft coupling — useful-energy demand (rc_therm demand + base-year
# anchor pin + one-sided pathway caps; RP/BB fossil exits now run in useful
# mode). Plain run-ids stay the headline; BEFORE this batch archive the r5
# bounds-mode outputs (they become the hard-linkage sensitivity):
#   mv results/runs results/runs_r5_bounds
# baseline_4.xlsx is unchanged since r3; the fresh scenario_name (workbook:
# baseline-2025-r7) keeps the import hygiene — a cached baseline-2025-r5 would
# carry the r5 corridor bounds into the soft-coupling cells.
assert cfg.scenario_name == "baseline-2025-r8", "stale inputs.xlsx — re-sync the repo"
assert cfg.carbon_price_passthrough, "W5.3 passthrough off — stale config"
assert cfg.price_damping < 1.0, "W5.8 price damping inactive — stale inputs.xlsx"
ref = pathways()["reference"]
print("ixmp platform:", cfg.platform_name)

# Open ONE ixmp platform and reuse it for every cell below. Opening a fresh
# platform per cell makes the HSQLDB file store fail to reopen partway through a
# batch (ClassCastException in the .script reload); one shared platform avoids it.
platform = build.get_platform(cfg)

# --- dynamic re-run machinery (§3i) -----------------------------------------
# Batch cells register their (Scenario, Config) per run-id; §3i re-solves only
# the ids whose latest runlog verdict is non-converged (alpha=0.3, budget 40).
import re as _re

from linkage.pipeline import run_id

RERUN_REGISTRY = {}

_RUNNING_RE = _re.compile(r"INFO (?:sturm_messageix|linkage)\.pipeline: Running (\S+)")

def run_verdicts(logfile):
    """Latest per-run-id convergence verdict from the runlog (True = converged)."""
    verdicts, current = {}, None
    for line in Path(logfile).read_text(encoding="utf-8", errors="replace").splitlines():
        m = _RUNNING_RE.search(line)
        if m:
            current = m.group(1)
        elif current and "Linkage converged" in line:
            verdicts[current] = True
        elif current and "Iteration budget" in line:
            verdicts[current] = False
    return verdicts

print("config:", cfg)
print("ready")

## 0 — Preflight: is the DATA in sync with the code?

The July runs were burned twice by partial syncs (new `.py`, stale data files).
This cell fails loudly if any critical data artefact predates the current code.


In [ ]:
# --- 0. Preflight data-sync sanity check (no solve) ---
import pandas as pd
from common import paths

# Restricted third-party inputs (not in the public repository; available on request):
paths.require_restricted(paths.BASELINE_XLSX, "MESSAGEix-Austria baseline workbook")
paths.require_restricted(paths.MACKO_XLSX, "Macko (2025) reduction tables")

problems = []
# (a) inputs.xlsx: REP-0989 factors + supply-dial pathway columns
xl = pd.ExcelFile(paths.INPUTS_XLSX)
ef = xl.parse("emission_factors").set_index("fuel")
if abs(ef.loc["biomass", "ef_direct"] - 0.015) > 1e-9:
    problems.append(f"inputs.xlsx emission_factors stale (biomass {ef.loc['biomass','ef_direct']}, want 0.015 = REP-0989 (values unchanged from REP-0948))")
pw = xl.parse("pathways")
for col in ("renewable_expansion", "fossil_power_exit_year"):
    if col not in pw.columns:
        problems.append(f"inputs.xlsx pathways sheet lacks {col!r} (supply dials would silently no-op)")
# (b) baseline_4: emission bake with REP-0989 values
b4 = pd.ExcelFile(paths.BASELINE_XLSX)
if "emission_factor" not in b4.sheet_names:
    problems.append("baseline_4.xlsx lacks the emission_factor sheet (re-bake missing)")
else:
    b4ef = b4.parse("emission_factor")
    bio = b4ef[(b4ef.technology == "biomass_rc") & (b4ef.year_act == 2025)]["value"]
    if bio.empty or abs(float(bio.iloc[0]) - 0.2022) > 0.005:
        problems.append(f"baseline_4 biomass_rc factor {None if bio.empty else round(float(bio.iloc[0]),4)} != ~0.2022 (REP-0989 re-bake missing)")
# (c) STURM manifest: the two columns the pipeline runs
manifest = (paths.STURM_DATA / "input_list_resid_SSP_2023.csv").open().readline()
if "SSP2" not in manifest.split(","):
    problems.append("STURM manifest lacks the SSP2 column (STURM data tree stale — restore the data tree from the repository)")
# (d) pathway records actually carry the dials
from common.scenarios import pathways as _pw
rp = _pw()["renewables_push"]
if rp.renewable_expansion is None or rp.fossil_power_exit_year is None:
    problems.append("renewables_push pathway has no supply dials (stale workbook wins over yaml)")
if "bio_bridge" not in _pw():
    problems.append("bio_bridge pathway missing")

# (e) W5.2: renovation-ambition variant columns + ceiling file
if "SSP2_RENAT" not in manifest.strip().split(","):
    problems.append("manifest lacks the SSP2_RENAT column (STURM data tree stale — restore the data tree from the repository)")
renmax = pd.read_csv(paths.STURM_DATA / "input_csv_SSP_2023_resid" / "ren_rate_en_max_ssp2_AT_TARGET.csv")
if renmax[renmax.year == 2030]["value"].iloc[0] != 0.03:
    problems.append("AT_TARGET renovation ceiling != 3%/yr at 2030")
if _pw()["renewables_push"].renovation_ambition != "at_target":
    problems.append("renewables_push lacks renovation_ambition=at_target (stale workbook)")
# (f) W5.3: activated switch stream (per-fuel AUT rows, no inert nsp)
sw = pd.read_csv(paths.STURM_DATA / "input_csv_SSP_2023_resid" / "rate_switch_fuel_heat_ssp2.csv")
sw_fuels = set(sw[sw.region_gea == "AUT"]["fuel_heat"])
if "nsp" in sw_fuels or not {"gas", "oil"} <= sw_fuels:
    problems.append(f"switch stream not activated (AUT fuels {sw_fuels}; STURM data tree stale — restore the data tree from the repository)")
# (g) W5.6: restructured fuel set + forks + elec_hp price rows
comb = pd.read_csv(paths.STURM_DATA / "input_csv_SSP_2023_resid" / "ct_fuel_comb.csv")
if "elec_hp" not in set(comb.fuel_heat):
    problems.append("ct_fuel_comb lacks elec_hp (STURM data tree stale — restore the data tree from the repository)")
dh_flag = comb[(comb.mat == "perm") & (comb.fuel_heat == "district_heat")]["mod_decision"]
if dh_flag.empty or int(dh_flag.iloc[0]) != 1:
    problems.append("district_heat not decision-modelled (STURM data tree stale — restore the data tree from the repository)")
for rfile in ("F06_stock_dyn_complete_rev.R", "F10_scenario_runs_MESSAGE_2100.R"):
    if "FORK W5.6" not in (paths.STURM_DATA.parent / "model" / rfile).read_text():
        problems.append(f"{rfile} lacks the W5.6 fork (partial sync)")
if "elec_hp" not in set(pd.read_csv(paths.STURM_DATA / "input_prices_R12.csv")["commodity"]):
    problems.append("input_prices_R12.csv lacks elec_hp rows (STURM data tree stale — restore the data tree from the repository)")
# (h) W5.1 foreign-region check now lives in the package (sturm.driver._require_no_foreign_rows,
# fires on the first STURM call) — the former notebook copy was removed 2026-09-04.
# (i) r7 demand-row reconciliation (changelog §22): rc_therm demand at 2025 must
# equal the base-year pin sum (internal closure, decision B1).
dem = pd.ExcelFile(paths.BASELINE_XLSX).parse("demand")
d25 = dem[(dem.commodity == "rc_therm") & (dem.year == 2025)]["value"].sum()
if abs(d25 - 8.3833) > 0.01:
    problems.append(f"baseline_4 rc_therm demand 2025 = {d25:.4f}, want 8.3833 (r7 bake missing?)")
# (j) r7 carbon-price path (decision B2): the Reference's legislated path must be
# complete — the workbook ships 2030–2040 blank until filled from REP-0995.
from common import workbook as _wbk
from common.scenarios import pathways as _pws
_ref = _pws()["reference"]
if _ref.carbon_price_path:
    try:
        _path = _wbk.carbon_price_path(_ref.carbon_price_path)
        missing = [y for y in (2025, 2030, 2035, 2040) if y not in _path]
        if missing:
            problems.append(f"carbon price path {_ref.carbon_price_path!r} lacks years {missing}")
    except (KeyError, ValueError) as exc:
        problems.append(str(exc))

if problems:
    raise SystemExit("DATA OUT OF SYNC:\n- " + "\n- ".join(problems))
print("preflight OK — data matches the code (REP-0989, supply dials, "
      "W5: prune + RENAT + switch rates + fuel set + forks)")


## 2 — Useful-mode probe: reference × baseline (convergence + base-year gate)

Iterative soft coupling (§17): STURM → `rc_therm` useful demand → solve →
**damped** price feedback (`price_damping`, W5.8) → repeat to convergence, on
top of the once-per-cell static anchor (base-year per-fuel calibration pin).

Run this **before the full columns**. Two gates:

1. **Convergence** — expect `Price feedback damping alpha = 0.5` in the log and
   `Linkage converged at iteration k` (the archived r5 useful probe converged in
   2; the anchor pin may shift that). A cell that still cycles is caught
   automatically by the §3i dynamic re-run (α = 0.3, budget 40).
2. **Base-year calibration** — `base_year_check.csv` must pass **per fuel** now
   (the anchor pin closes the 6.6 Mt anomaly's base-year half: the archived
   unpinned useful probe had gas +78 % / biomass −64 % and 2025 CO₂ 8,888 kt vs
   the 5,574 kt canon). The cell also prints 2025/2040 buildings CO₂ next to the
   r5 bounds canon (`results/runs_r5_bounds`) — the 2040 **gap is expected**
   (soft coupling transmits no STURM price-driven fuel switching); record it,
   don't "fix" it.

In [ ]:
run_scenario(Scenario(pathway=ref, digitalization=digitalization_levels()["baseline"]), cfg, platform=platform)

# W5.8 probe gate: the damped loop must actually CONVERGE, not fall back to the
# oscillation mean like the 2026-07-10/12 batches did.
def probe_check(mode, rid):
    # Scoped to the run-id via run_verdicts (cell 1) — the previous version
    # read the globally LAST verdict line, correct only by cell-ordering luck.
    verdicts = run_verdicts(LOGFILE)
    assert rid in verdicts, f"no convergence trace for {rid} — did the cell run?"
    assert verdicts[rid], (
        f"{mode} probe still cycling — re-run this cell with "
        "dataclasses.replace(cfg, price_damping=0.3) before launching the batch"
    )
    print(f"{rid}: converged")

probe_check("useful", "pathway-reference__digi-baseline")

# §17 base-year gate: the anchor pin must restore the per-fuel calibration that
# the unpinned useful probe (2026-07-13 archive) failed.
import pandas as pd
from common.paths import RESULTS_RUNS

ref_dir = RESULTS_RUNS / "pathway-reference__digi-baseline"
byc = pd.read_csv(ref_dir / "base_year_check.csv")
print("\nbase_year_check:")
print(byc.to_string(index=False))
assert byc["passed"].all(), (
    "base-year calibration failing in useful mode — the anchor pin did not bind "
    "(stale scenario cache? bump scenario_name / fresh platform)"
)

def _co2(d, year):
    c = pd.read_csv(d / "buildings_co2.csv")
    v = c[c.year == year]["value"]
    return float(v.iloc[0]) if not v.empty else float("nan")

print("\nbuildings CO2 (kt), this run:")
for y in (2025, 2040):
    print(f"  {y}: {_co2(ref_dir, y):8.1f}")
print("(2025 must equal 5,589.8 kt by construction of the pin; 2040 = 905.2 kt in r8)")
print("baseline probe done")

## 2b — Useful-mode probe: Renewables Push × baseline (fossil-exit gate, NEW §17)

First-ever fossil-exit pathway in useful mode: the exit is imposed as **one-sided
static caps** (`apply_useful_anchor`: gas → 0 at 2040, oil/coal at 2035, logistic
ramp) on top of the free fuel mix, with the baseline's dynamic constraints
stripped over the projection years so the absorbing techs (heat pump, DH,
electric) can cover the demand the capped fuels drop. Gates: **feasible solve**
(an infeasibility here means the dynamics strip missed a colliding constraint —
first suspect `heat_rc`'s upstream chain), fossil activity zero by the ban
years, and convergence.

In [ ]:
# --- 2b. RP × baseline probe: fossil-exit caps in useful mode (§17 gate) ---
rp = pathways()["renewables_push"]
run_scenario(Scenario(pathway=rp, digitalization=digitalization_levels()["baseline"]), cfg, platform=platform)

probe_check("useful/renewables_push", "pathway-renewables_push__digi-baseline")

rp_dir = RESULTS_RUNS / "pathway-renewables_push__digi-baseline"
act = pd.read_csv(rp_dir / "activity_rc.csv")

def _act(tech, year):
    v = act[(act.year == year) & (act.technology == tech)]["value"]
    return float(v.iloc[0]) if not v.empty else 0.0

print("fossil end-use activity (GWa):")
for tech, ban in (("gas_rc", 2040), ("loil_rc", 2035), ("coal_rc", 2035)):
    vals = {y: _act(tech, y) for y in (2030, 2035, 2040)}
    print(f"  {tech:9s} {vals}")
    for y, v in vals.items():
        if y >= ban:
            assert v <= 1e-6, f"{tech} not zero at {y} (ban {ban}) — caps did not bind"

byc = pd.read_csv(rp_dir / "base_year_check.csv")
assert byc["passed"].all(), "RP base-year calibration failing"
print(f"\n2040 buildings CO2 (kt): {_co2(rp_dir, 2040):.1f} "
      f"(reference probe: {_co2(ref_dir, 2040):.1f} — RP must be well below)")
print("renewables_push probe done — batch is GO")

## 3 — Full reference column (all 3 digitalization levels) → RQ2
Run this to get the digitalization comparison for the Reference pathway. (The constrained pathways are now implemented — section 3b runs Renewables Push; use `run_matrix(cfg)` for the full 3×3.)

In [ ]:
# Reuse the single shared platform for all three cells; register for §3i.
scens = [Scenario(pathway=ref, digitalization=d) for d in digitalization_levels().values()]
RERUN_REGISTRY.update({run_id(s, cfg): (s, cfg) for s in scens})
run_scenarios(scens, cfg, platform=platform)
print("reference column done")

## 3b — Renewables Push column → RQ1 second pathway

Runs the **Renewables Push** pathway (fossil exit — gas 2040, oil/coal 2035; high electrification; low bioenergy) across all three digitalization levels — the second pathway for the RQ1 gap-to-target comparison.

**Since §17 (useful mode)** the fossil exit is imposed as one-sided static caps by `apply_useful_anchor` (gate: the §2b probe), the bioenergy `low` dial caps `biomass_rc` at base level, and `electrification_intensity=high` is a structural no-op (fuel mix free by construction — the loop logs this). If a cell comes back **infeasible**, the caps + demand collided with a remaining supply-side constraint — check `heat_rc`'s upstream chain first (fallback ladder: relax the colliding constraint one step at a time, upstream-first, and record each step in the run log).

In [ ]:
# --- Renewables Push column (all 3 digitalization levels) → second RQ1 pathway ---
# Reuses the single shared `platform` from cell 1 (avoids the HSQLDB reopen corruption).
rp = pathways()["renewables_push"]
scens = [Scenario(pathway=rp, digitalization=d) for d in digitalization_levels().values()]
RERUN_REGISTRY.update({run_id(s, cfg): (s, cfg) for s in scens})
run_scenarios(scens, cfg, platform=platform)
print("renewables_push column done")

## 3g — Bio-Bridge column (completes the 3×3 matrix)

High bioenergy, fossil exit 2040, `eag2030` renewable
floors. First solve of this pathway — watch for infeasibility in the log.


In [ ]:
# --- 3g. Bio-Bridge column (all 3 digitalization levels) ---
bb = pathways()["bio_bridge"]
scens = [Scenario(pathway=bb, digitalization=d) for d in digitalization_levels().values()]
RERUN_REGISTRY.update({run_id(s, cfg): (s, cfg) for s in scens})
run_scenarios(scens, cfg, platform=platform)
print("bio_bridge column done — 3x3 matrix complete")

## 3d — Renewables Push, LINEAR fossil-exit shape (sensitivity)

Same three RP cells with `fossil_exit_shape="linear"` (the sensitivity-branch
comparator; logistic is the default). Outputs land in separate
`pathway-renewables_push__digi-*__shape-linear/` folders (`pipeline.run_id`),
so the matrix folders are never overwritten, and the viz collectors ignore them.


In [ ]:
# --- 3d. RP column with the LINEAR fossil-exit shape (sensitivity) ---
import dataclasses

cfg_linear = dataclasses.replace(cfg, fossil_exit_shape="linear")
rp = pathways()["renewables_push"]
scens = [Scenario(pathway=rp, digitalization=d) for d in digitalization_levels().values()]
RERUN_REGISTRY.update({run_id(s, cfg_linear): (s, cfg_linear) for s in scens})
run_scenarios(scens, cfg_linear, platform=platform)
print("renewables_push LINEAR sensitivity column done (…__shape-linear folders)")

## 3c. Carbon-price ladder (r7 B2)

Five flat-price Reference cells (0/40/55/80/120 USD/tCO₂). The switching
threshold where gas loses to heat pumps (reviews: ~30–60 USD given the 6–12%
operating-cost margin) is itself a finding; the legislated-path Reference in §3
is the headline, these bracket it. tax0 preserves the pre-r7 zero-price
Reference as the ladder origin.

In [ ]:
# --- 3c. Carbon-price ladder (r7 B2): flat Reference cells to locate the
# gas->heat-pump switching threshold. The headline Reference (§3) carries the
# legislated NEHG/ETS2 path from the workbook; these flat cells bracket it.
# tax0 = the pre-r7 zero-price Reference, kept as the ladder origin.
import dataclasses

ref = pathways()["reference"]
base = digitalization_levels()["baseline"]
for level in (0, 40, 55, 80, 120):
    p = dataclasses.replace(
        ref, id=f"ref_tax{level}", label=f"Reference + {level} USD/tCO2 (flat)",
        carbon_price=float(level), carbon_price_path=None,
    )
    s = Scenario(pathway=p, digitalization=base)
    RERUN_REGISTRY[run_id(s, cfg)] = (s, cfg)
    run_scenario(s, cfg, platform=platform)

import pandas as pd
print("\nladder 2040 buildings CO2 (kt):")
for level in (0, 40, 55, 80, 120):
    co2 = pd.read_csv(RESULTS_RUNS / f"pathway-ref_tax{level}__digi-baseline" / "buildings_co2.csv")
    print(f"  {level:>4} USD/t: {co2.loc[co2.year == 2040, 'value'].iloc[0]:9.1f}")


## 3w. Endogenous Wilson band (r7 B8)

Four Reference cells with the imposed `rc_therm` demand scaled by the Wilson,
Zakeri et al. (2026) buildings modifiers (−22/−5/+3/+9%, ramped 2025→2050).
Replaces the ex-post multiplication (`viz/digitalization_band.py` reads these
folders automatically) so the band carries the model's real demand→emissions
elasticity and is comparable to the RQ2 spread.

In [ ]:
# --- 3w. Endogenous Wilson-band cells (r7 B8) ---
import dataclasses

WILSON = {"enable_extreme": -0.22, "enable_cautious": -0.05,
          "undermine_cautious": +0.03, "undermine_extreme": +0.09}
ref = pathways()["reference"]
base = digitalization_levels()["baseline"]
for name, m in WILSON.items():
    p = dataclasses.replace(
        ref, id=f"ref_wilson_{name}", label=f"Reference, Wilson {name} ({m:+.0%})",
        demand_modifier=m,
    )
    s = Scenario(pathway=p, digitalization=base)
    RERUN_REGISTRY[run_id(s, cfg)] = (s, cfg)
    run_scenario(s, cfg, platform=platform)
print("wilson cells done")

## 3n. No-digitalization counterfactual (r7 B6)

One Reference cell on the plain `SSP2` operating-hours column (flat 5.0 h/day)
— the digitalization-vs-none comparison the adoption-rate matrix cannot give.

In [ ]:
# --- 3n. No-DIG counterfactual (r7 B6) ---
from common.scenarios import Digitalization

none = Digitalization(id="none", label="No digitalization (SSP2)",
                      macko_column="Baseline_Digitalization")  # column unused in sturm mode
s = Scenario(pathway=pathways()["reference"], digitalization=none)
RERUN_REGISTRY[run_id(s, cfg)] = (s, cfg)
run_scenario(s, cfg, platform=platform)

import pandas as pd
co2 = pd.read_csv(RESULTS_RUNS / "pathway-reference__digi-none" / "buildings_co2.csv")
print("no-DIG Reference 2040:", round(co2.loc[co2.year == 2040, "value"].iloc[0], 1), "kt")

## 3i — Dynamic convergence re-run (α = 0.3, budget 40, failures only)

The first pass runs at the workbook defaults (α = 0.5, budget 20): historically
converging cells finish in ≤ 9 iterations, and the hard period-2 cyclers fail
fast instead of burning a doubled budget on a flat cycle. This cell re-runs
**only** the registered cells whose latest runlog verdict is non-converged, at
`price_damping = 0.3` with `max_iterations = 40` — the 2026-07-13 traces show
slow decay at 0.3 (final L-inf 0.011–0.021 at budget 20, trending down), so the
doubled budget closes the remaining gap. Per-cell α lands in
`convergence_evidence.csv`, so mixed damping across cells stays transparent.
If a cell still fails: drop to α = 0.2 (`dataclasses.replace`) or report its
residual L-inf as the oscillation band.

In [ ]:
# --- 3i. Dynamic re-run of non-converged cells (alpha=0.3, budget 40; W5.8b) ---
import dataclasses

assert RERUN_REGISTRY, (
    "RERUN_REGISTRY is empty — cell 1 was re-executed after the batch (fresh "
    "logfile + cleared registry), so this cell would vacuously pass. Re-run the "
    "batch cells (or restore LOGFILE/RERUN_REGISTRY) before §3i."
)
verdicts = run_verdicts(LOGFILE)
todo = [rid for rid, ok in verdicts.items() if not ok and rid in RERUN_REGISTRY]
unregistered = [rid for rid, ok in verdicts.items() if not ok and rid not in RERUN_REGISTRY]
if unregistered:
    print("non-converged but not registered this session (ignored):", unregistered)
print(f"{len(todo)} cell(s) to re-run at price_damping=0.3, max_iterations=40:")
for rid in todo:
    print("  ", rid)
for rid in todo:
    scen, cfg_i = RERUN_REGISTRY[rid]
    run_scenario(scen, dataclasses.replace(cfg_i, price_damping=0.3, max_iterations=40),
                 platform=platform)

final = run_verdicts(LOGFILE)
print("\nfinal verdicts (registered cells):")
for rid in sorted(RERUN_REGISTRY):
    print(f"  {'PASS' if final.get(rid) else 'FAIL':4s} {rid}")
failed = [rid for rid in RERUN_REGISTRY if not final.get(rid, False)]
assert not failed, (
    f"still not converged after the 0.3/40 re-run: {failed} — drop to "
    "price_damping=0.2 for these cells (dataclasses.replace), or report the "
    "residual L-inf as the oscillation band."
)
print("all registered cells converged")

## 4 — Inspect results

In [ ]:
from common.paths import RESULTS_RUNS
import pandas as pd
for p in sorted(RESULTS_RUNS.glob("pathway-*__digi-*")):
    print(p.name, "->", sorted(f.name for f in p.iterdir()))

# RQ1 headline: 2040 buildings CO2 (kt) per cell. Matrix cells only — the
# ref_tax120 probe and __shape/__mode/__once variants are listed separately.
import re as _re
_MATRIX = _re.compile(r"^pathway-(reference|renewables_push|bio_bridge)__digi-[a-z]+$")
print("\n2040 buildings CO2 (kt) by matrix cell (sensitivities marked *):")
for p in sorted(RESULTS_RUNS.glob("pathway-*__digi-*")):
    mark = "" if _MATRIX.match(p.name) else "*"
    co2 = pd.read_csv(p / "buildings_co2.csv")
    v = co2.loc[co2["year"] == 2040, "value"]
    if not v.empty:
        print(f"  {p.name+mark:47s} {v.iloc[0]:8.1f}")

base = RESULTS_RUNS / "pathway-reference__digi-baseline"
print("\nreference/baseline base_year_check:")
print(pd.read_csv(base / "base_year_check.csv").to_string(index=False))

## 5 — Visualisations
Renders the figures with `viz/plots.R` (ggplot2) from `results/runs/*` and shows them inline: emissions trajectory vs the 2040 target (RQ1), buildings fuel mix (RQ2), base-year (2025) calibration check vs Statistik Austria (2024), and system final energy. Needs R with `ggplot2, readr, dplyr, tidyr, scales` (see `viz/README.md`). PNGs + a combined `figures.pdf` also land in `results/figures/`.

In [ ]:
# Render the figures (viz/plots.R, ggplot2) from the run outputs and show them inline.
import subprocess
import sys
from pathlib import Path
from IPython.display import Image, display

# Every table/figure script the thesis relies on (see README "Reproducing the thesis
# tables and figures"); a failing script stops the workflow instead of leaving a stale
# committed table in place.
for script in ("viz/summary_tables.py", "viz/convergence_table.py",
               "viz/digitalization_band.py",      # Wilson demand band (table + figure)
               "viz/ladder_table.py",             # carbon-price ladder
               "viz/shape_sensitivity.py",        # fossil-exit shape (table + figure)
               "viz/buildings_final_energy.py",   # final energy by fuel, decomposition, emission factors
               "viz/adoption_differences.py"):    # adoption differences (table + figure)
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-500:])
    if r.returncode != 0:
        raise RuntimeError(f"{script} failed (exit {r.returncode}):\n{r.stderr[-2000:]}")

FIG_DIR = Path("results/figures")
res = subprocess.run(
    ["Rscript", "viz/plots.R", "results/runs", str(FIG_DIR)],
    capture_output=True, text=True,
)
print(res.stdout or "(no stdout)")
if res.returncode != 0:
    raise RuntimeError("plots.R failed — install the R plotting packages (see viz/README.md):\n" + res.stderr[-1000:])
for png in sorted(FIG_DIR.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## 6 — Capture the run for review
Download `results/logs/run_*.log` (robust), or run the cell below to also export this notebook to HTML (save the notebook first).

In [ ]:
print("run log:", LOGFILE)
print("-" * 80)
print(LOGFILE.read_text(encoding="utf-8")[-4000:])

import subprocess, sys
res = subprocess.run(
    [sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", "--output-dir", "results", "run.ipynb"],
    capture_output=True, text=True,
)
print("\nnbconvert:", "ok -> results/run.html" if res.returncode == 0 else f"failed; send results/logs/*.log instead\n{res.stderr[-400:]}")